In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

In [6]:
df = pd.read_csv("../DATA/PROCESS/save_dataset3.csv")

In [9]:
df.drop(columns=["Unnamed: 0.1", "Unnamed: 0"], inplace=True)

## Random Test

In [10]:
# 1. 기본 세팅

target_col = "target"

drop_cols = [
    target_col,
    "datetime",
]

drop_cols = [col for col in drop_cols if col in df.columns]

X = df.drop(columns=drop_cols)
y = df[target_col]

print("X shape:", X.shape)
print("y distribution:")
print(y.value_counts(normalize=True))


# 2. 컬럼 타입 분리


categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

print("numeric cols:", len(numeric_cols))
print("categorical cols:", len(categorical_cols))

# 3. 전처리 파이프라인

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

# 4. Train/Test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("train target distribution:")
print(y_train.value_counts(normalize=True))

print("test target distribution:")
print(y_test.value_counts(normalize=True))

# 5. 모델 학습

model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

# 6. 예측 및 평가

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("\nClassification Report")
print(classification_report(y_test, y_pred, digits=4))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nROC-AUC")
print(roc_auc_score(y_test, y_proba))

X shape: (369754, 90)
y distribution:
target
0    0.544673
1    0.455327
Name: proportion, dtype: float64
numeric cols: 89
categorical cols: 1


/var/folders/g8/gwqmqng10_g3h7m8r02yq1qc0000gn/T/ipykernel_58289/1362361736.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()


train target distribution:
target
0    0.544673
1    0.455327
Name: proportion, dtype: float64
test target distribution:
target
0    0.544671
1    0.455329
Name: proportion, dtype: float64

Classification Report
              precision    recall  f1-score   support

           0     0.9163    0.9215    0.9189     40279
           1     0.9055    0.8993    0.9024     33672

    accuracy                         0.9114     73951
   macro avg     0.9109    0.9104    0.9106     73951
weighted avg     0.9114    0.9114    0.9114     73951


Confusion Matrix
[[37118  3161]
 [ 3391 30281]]

ROC-AUC
0.9696863268727948


In [12]:
# 7. Feature Importance 확인

rf = model.named_steps["clf"]

feature_names = model.named_steps["preprocess"].get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

importance_df.head(30)

,feature,importance
3,num__hour,0.127907
7,num__hour_cos,0.078663
6,num__hour_sin,0.063019
42,num__rest_cnt_500m,0.054801
43,num__rest_cnt_1000m,0.029487
41,num__rest_cnt_300m,0.029143
85,num__subway_weighted_access_k5,0.027568
86,num__subway_mean_dist_k5_m,0.022304
12,num__gu_walk_area,0.019071
53,num__tour_food_mean_dist_k20_m,0.017638


## Time Test

In [13]:
df = df.sort_values("datetime")

split_idx = int(len(df) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

In [14]:
print("train target distribution:")
print(y_train.value_counts(normalize=True))

print("test target distribution:")
print(y_test.value_counts(normalize=True))

# 5. 모델 학습

model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

# 6. 예측 및 평가

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("\nClassification Report")
print(classification_report(y_test, y_pred, digits=4))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nROC-AUC")
print(roc_auc_score(y_test, y_proba))

train target distribution:
target
0    0.578189
1    0.421811
Name: proportion, dtype: float64
test target distribution:
target
1    0.58939
0    0.41061
Name: proportion, dtype: float64

Classification Report
              precision    recall  f1-score   support

           0     0.3943    0.7289    0.5117     30365
           1     0.5378    0.2198    0.3120     43586

    accuracy                         0.4288     73951
   macro avg     0.4661    0.4744    0.4119     73951
weighted avg     0.4789    0.4288    0.3940     73951


Confusion Matrix
[[22134  8231]
 [34007  9579]]

ROC-AUC
0.6599856709035162


In [15]:
# 7. Feature Importance 확인

rf = model.named_steps["clf"]

feature_names = model.named_steps["preprocess"].get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

importance_df.head(30)

,feature,importance
3,num__hour,0.106530
7,num__hour_cos,0.061186
6,num__hour_sin,0.053349
42,num__rest_cnt_500m,0.050201
43,num__rest_cnt_1000m,0.038357
53,num__tour_food_mean_dist_k20_m,0.036078
85,num__subway_weighted_access_k5,0.025511
62,num__shop_mean_dist_k20_m,0.023320
41,num__rest_cnt_300m,0.022890
86,num__subway_mean_dist_k5_m,0.022373
